# Lab 1a · Un agente desde cero (sin frameworks)

**Objetivo:** entender qué hay *dentro* de un agente programando su bucle a mano: **modelo + herramientas + bucle**.

```
pregunta ──► MODELO ──¿pide una herramienta?──► sí ──► ejecutamos la herramienta ──► resultado ──┐
               ▲                                                                                │
               └────────────────────────────────────────────────────────────────────────────────┘
                                     no ──► respuesta final
```

Las celdas marcadas con **✏️ TODO** son para ti. Tiempo estimado: 20 minutos.

In [ ]:
import json

import mucdia
from mucdia import negocio

cliente = mucdia.cliente_openai()  # misma interfaz que el SDK oficial de OpenAI, apuntando a Gemini
MODELO = mucdia.config.modelo()
print("Modelo:", MODELO)

## Paso 1 · Un modelo sin herramientas

Preguntemos algo que solo se puede responder con nuestros datos.

In [ ]:
pregunta = "¿Cuánto vendió MUCDIA en septiembre de 2026? Dame la cifra exacta."
respuesta = cliente.chat.completions.create(model=MODELO, messages=[{"role": "user", "content": pregunta}])
print(respuesta.choices[0].message.content)

¿Te fías de esa respuesta? El modelo **no tiene acceso a nuestros datos**: o se niega, o se inventa una cifra
(*alucina*). Para que pueda actuar sobre el mundo necesita **herramientas**.

## Paso 2 · Las herramientas son funciones normales

En `mucdia/negocio.py` hay funciones de Python que consultan la base de datos. Pruébalas directamente:

In [ ]:
print(negocio.ventas_por_tienda("2026-09-01", "2026-09-30"))

## Paso 3 · Describir las herramientas al modelo

El modelo no ve el código: solo ve un **nombre**, una **descripción** y los **parámetros** (en JSON Schema).
Con eso decide *cuándo* y *cómo* usar cada herramienta. Una buena descripción es la mitad del trabajo.

**✏️ TODO:** completa la descripción de `top_productos` (mira su firma: `desde`, `hasta`, `n=5`, `categoria=None`).

In [ ]:
FECHA = {"type": "string", "description": "Fecha en formato AAAA-MM-DD"}
CATEGORIA = {"type": "string", "description": "Electrónica, Hogar, Deporte, Moda y complementos, Belleza y cuidado o Gourmet"}

HERRAMIENTAS = [
    {"type": "function", "function": {
        "name": "ventas_por_categoria",
        "description": "Ventas netas (€), pedidos y unidades por categoría entre dos fechas incluidas. Solo pedidos completados.",
        "parameters": {"type": "object", "properties": {"desde": FECHA, "hasta": FECHA}, "required": ["desde", "hasta"]},
    }},
    {"type": "function", "function": {
        "name": "ventas_por_tienda",
        "description": "Ventas netas (€) por tienda (T01-T08 físicas, WEB = online) entre dos fechas, opcionalmente de una categoría.",
        "parameters": {"type": "object", "properties": {"desde": FECHA, "hasta": FECHA, "categoria": CATEGORIA},
                       "required": ["desde", "hasta"]},
    }},
    {"type": "function", "function": {
        "name": "top_productos",
        "description": "...",  # ✏️ TODO: ¿qué devuelve? ¿cuándo debería usarla el modelo?
        "parameters": {"type": "object", "properties": {
            "desde": FECHA, "hasta": FECHA,
            # ✏️ TODO: añade "n" (integer) y "categoria" (puedes reutilizar CATEGORIA)
        }, "required": ["desde", "hasta"]},
    }},
]

FUNCIONES = {
    "ventas_por_categoria": negocio.ventas_por_categoria,
    "ventas_por_tienda": negocio.ventas_por_tienda,
    "top_productos": negocio.top_productos,
}

## Paso 4 · El modelo *pide* una herramienta… pero no la ejecuta

In [ ]:
SISTEMA = """Eres el analista de datos de MUCDIA S.A. (una cadena ficticia de tiendas).
Usa las herramientas para obtener cifras reales: nunca te inventes datos.
Hay datos del 2025-01-01 al 2026-10-31. Responde en español, de forma breve y con cifras."""

mensajes = [
    {"role": "system", "content": SISTEMA},
    {"role": "user", "content": "¿Cuáles fueron los 3 productos más vendidos durante el Black Friday 2025 (21 nov - 1 dic)?"},
]
respuesta = cliente.chat.completions.create(model=MODELO, messages=mensajes, tools=HERRAMIENTAS)
mensaje = respuesta.choices[0].message
print("Texto del modelo:", mensaje.content)
for llamada in mensaje.tool_calls or []:
    print("🔧 Pide ejecutar:", llamada.function.name, llamada.function.arguments)

El modelo devuelve una **intención**: el nombre de la herramienta y sus argumentos en JSON.
**Quien ejecuta es nuestro código.** Ahí es donde se pone el control (lo veremos en la sesión 3).

## Paso 5 · El bucle agéntico ✏️

Completa la función: **pensar** (llamar al modelo) → **¿ha terminado?** → **actuar** (ejecutar herramientas) →
**observar** (devolverle los resultados) → repetir.

> 💡 Guarda el mensaje del modelo **tal cual** (`mensajes.append(mensaje)`). Gemini 3 incluye en él una
> *firma de razonamiento* que necesita en la siguiente vuelta; si lo reconstruyes a mano, la API devuelve error 400.

In [ ]:
def ejecutar_agente(pregunta, max_pasos=6, verbose=True):
    mensajes = [{"role": "system", "content": SISTEMA}, {"role": "user", "content": pregunta}]
    for paso in range(1, max_pasos + 1):
        # 1) PENSAR. ✏️ TODO: llama al modelo con MODELO, mensajes y HERRAMIENTAS (como en el paso 4)
        respuesta = ...
        mensaje = respuesta.choices[0].message

        # 2) ¿HA TERMINADO? ✏️ TODO: si el modelo no pide herramientas (mensaje.tool_calls vacío), devuelve su texto
        ...

        # 3) Guardamos su mensaje tal cual (incluye la firma de razonamiento de Gemini 3)
        mensajes.append(mensaje)

        # 4) ACTUAR y 5) OBSERVAR
        for llamada in mensaje.tool_calls:
            nombre = llamada.function.name
            argumentos = json.loads(llamada.function.arguments or "{}")
            if verbose:
                print(f"🔧 Paso {paso}: {nombre}({argumentos})")
            try:
                resultado = ...  # ✏️ TODO: ejecuta la función correcta de FUNCIONES con **argumentos
            except Exception as error:  # el modelo también aprende de los errores
                resultado = f"ERROR al ejecutar {nombre}: {error}"
            mensajes.append({"role": "tool", "tool_call_id": llamada.id, "content": str(resultado)})
    return "⚠️ He alcanzado el máximo de pasos sin una respuesta final."

In [ ]:
print(ejecutar_agente(
    "¿Qué categoría vendió más en el tercer trimestre de 2026 y qué tienda vendió más de esa categoría? "
    "Compáralo con el tercer trimestre de 2025."
))

🎉 Acabas de programar un agente: el modelo decide **qué** herramientas usar, **en qué orden** y **cuándo parar**.
Fíjate en que puede pedir **varias herramientas en el mismo paso** (en paralelo).

## Paso 6 · Rómpelo (para entenderlo)

**a) Pregunta ambigua.** ¿Qué supone el agente? ¿Pregunta o se lanza?

In [ ]:
print(ejecutar_agente("¿Qué tal fue el verano?"))

**b) Una herramienta que falla.** Simulamos una caída de la base de datos. ¿Se recupera el agente? ¿Es honesto?

In [ ]:
original = FUNCIONES["ventas_por_categoria"]
FUNCIONES["ventas_por_categoria"] = lambda **kwargs: "ERROR: la base de datos no responde (timeout)."
print(ejecutar_agente("¿Qué categoría vendió más en octubre de 2026?"))
FUNCIONES["ventas_por_categoria"] = original

**c) Una descripción engañosa.** El modelo solo "ve" las descripciones. ¿Qué pasa si mienten?

In [ ]:
import copy

copia = copy.deepcopy(HERRAMIENTAS)
HERRAMIENTAS[1]["function"]["description"] = "Devuelve la previsión meteorológica de cada tienda."
print(ejecutar_agente("¿Qué tienda vendió más en octubre de 2026?"))
HERRAMIENTAS = copia

**d) Sin margen para pensar.** ¿Qué ocurre con un solo paso?

In [ ]:
print(ejecutar_agente("Compara las ventas de Hogar en Andalucía en el T3 de 2025 y el T3 de 2026.", max_pasos=1))

## Para pensar

1. ¿Quién **decide** qué herramienta usar? ¿Quién la **ejecuta**? ¿Dónde pondrías controles?
2. ¿Cuántas llamadas al modelo ha hecho cada pregunta? Mira `mucdia.uso()`.
3. Si MUCDIA recibiera 10.000 preguntas al día, ¿cuánto costaría al mes en el plan de pago?
4. ¿Qué tendrías que añadir para que esto funcionara en producción (errores, límites, memoria, trazas…)?
   👉 Eso es lo que nos da un **framework**: lo verás en el **Lab 1b**.

In [ ]:
mucdia.uso()